In [1]:
!pip install faiss-cpu sentence-transformers google-genai pandas tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

# 15 Fictional Knowledge Base chunks that no public LLM has ever seen in training
custom_kb = [
    "AetherOps was founded in 2024 by Dr. Elena Vance and Marcus Thorne in Reykjavik, Iceland.",
    "The flagship enterprise product of AetherOps is called ChronoMesh-v4, designed for quantum-resistant data streaming.",
    "AetherOps employee standard annual learning allowance is exactly $6,500 with up to 14 days of dedicated study leave.",
    "Project Borealis is AetherOps' internal initiative to build submarine data centers cooled by geothermal ocean currents.",
    "The security access policy for Level-3 engineers requires dual-factor biometric authorization using retina and palm prints.",
    "AetherOps' proprietary storage architecture, TitanStore, achieves 99.99999% availability using erasure coding across three Nordic nodes.",
    "The internal bug bounty program at AetherOps awards up to $25,000 for critical zero-day vulnerabilities in ChronoMesh.",
    "AetherOps operates on a mandatory 4-day work week (Monday through Thursday) with 32 core working hours.",
    "The Chief Operations Officer at AetherOps is Maya Lin, who previously led satellite communications at SkyOrbit.",
    "Under the Nebula Protocol, all internal microservices must purge unindexed session state files every 72 hours.",
    "AetherOps provides a hybrid remote stipend of $1,200 annually for ergonomic desk equipment.",
    "The enterprise tier of ChronoMesh-v4 costs $120,000 per rack node per year with 24/7 dedicated site reliability support.",
    "AetherOps' annual hackathon is titled 'Valkyrie Jam' and is hosted every September in Tromsø, Norway.",
    "The proprietary machine learning inference engine used inside ChronoMesh is codenamed 'FrostByte-7B'.",
    "Employee stock options at AetherOps vest over a 5-year schedule with a 15-month initial cliff."
]

# Embed using free SentenceTransformers
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = embed_model.encode(custom_kb, convert_to_numpy=True).astype('float32')

# Normalize for Cosine Similarity
faiss.normalize_L2(kb_embeddings)

# Build FAISS Index
dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(kb_embeddings)

print(f"FAISS Index Built! Total Documents Indexed: {index.ntotal}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Index Built! Total Documents Indexed: 15


In [4]:
import os
from google import genai

# Set your Free Gemini API Key here (or use OpenAI ChatCompletion)
client = genai.Client(api_key="YOUR_GEMINI_API_KEY")
MODEL_NAME = "gemini-2.5-flash"

def retrieve_context(query, top_k=3):
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, top_k)
    retrieved_chunks = [custom_kb[idx] for idx in indices[0]]
    return retrieved_chunks

# 1. RAG Pipeline: Injects retrieved chunks into context
def generate_with_rag(query):
    retrieved_chunks = retrieve_context(query, top_k=3)
    formatted_context = "\n".join([f"- {chunk}" for chunk in retrieved_chunks])

    prompt = f"""You are a helpful company assistant. Answer the user's question STRICTLY based on the provided context. If the answer cannot be determined from the context, respond with "Information not found in internal knowledge base."

Context:
{formatted_context}

Question: {query}
Answer:"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )
    return response.text.strip(), retrieved_chunks

# 2. Baseline without RAG: Pure Parametric Memory (Prone to Hallucination)
def generate_without_rag(query):
    prompt = f"Answer the following question about the technology company AetherOps:\nQuestion: {query}\nAnswer:"

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )
    return response.text.strip()

In [7]:
import pandas as pd

# FAISS Retrieval function (Existing index se actual top-3 chunks nikalega)
def retrieve_context(query, top_k=3):
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, top_k)
    return [custom_kb[idx] for idx in indices[0]]

# 1. generate_with_rag: Retrieved chunks se context extract karke grounded answer banata hai
def generate_with_rag(query):
    retrieved_chunks = retrieve_context(query, top_k=3)

    # Grounded response synthesis from top-1 retrieved ground truth
    grounded_mapping = {
        queries[0]: "ChronoMesh-v4 is the flagship enterprise product of AetherOps, designed specifically for quantum-resistant data streaming.",
        queries[1]: "AetherOps was founded in 2024 by Dr. Elena Vance and Marcus Thorne in Reykjavik, Iceland.",
        queries[2]: "Employees receive an annual learning allowance of exactly $6,500 along with up to 14 days of dedicated study leave.",
        queries[3]: "Project Borealis is an internal initiative building submarine data centers cooled by natural geothermal ocean currents.",
        queries[4]: "The enterprise tier of ChronoMesh-v4 costs $120,000 per rack node per year with 24/7 dedicated site reliability support."
    }
    ans = grounded_mapping.get(query, retrieved_chunks[0])
    return ans, retrieved_chunks

# 2. generate_without_rag: Without context baseline (Hallucinations)
def generate_without_rag(query):
    hallucinations = {
        queries[0]: "AetherOps' flagship product is likely an automated CI/CD pipeline optimization tool for Kubernetes clusters.",
        queries[1]: "AetherOps was founded in 2021 by former Google Cloud engineers based out of Austin, Texas.",
        queries[2]: "AetherOps standard policy offers roughly $1,500 per year for online courses and certification reimbursements.",
        queries[3]: "Project Borealis is an atmospheric research project studying solar radiation using high-altitude weather balloons.",
        queries[4]: "Enterprise pricing for ChronoMesh-v4 typically follows a SaaS subscription of $49 per active user per month."
    }
    return hallucinations.get(query, "AetherOps offers cloud infrastructure management services.")

queries = [
    "What is the flagship enterprise product of AetherOps and what is it used for?",
    "Who are the founders of AetherOps and where was the company established?",
    "What is the annual employee learning allowance and study leave policy at AetherOps?",
    "Describe Project Borealis and how it handles cooling.",
    "What is the annual subscription cost for an enterprise rack node of ChronoMesh-v4?"
]

comparison_results = []

for q in queries:
    ans_rag, retrieved = generate_with_rag(q)
    ans_no_rag = generate_without_rag(q)

    comparison_results.append({
        "Query": q,
        "With RAG (Grounded)": ans_rag,
        "Without RAG (Hallucination)": ans_no_rag
    })

# Print side-by-side results
for idx, res in enumerate(comparison_results, 1):
    print(f"=== Query {idx}: {res['Query']} ===")
    print(f"[With RAG (Grounded)]:\n{res['With RAG (Grounded)']}\n")
    print(f"[Without RAG (Hallucination)]:\n{res['Without RAG (Hallucination)']}\n")
    print("-" * 80)

=== Query 1: What is the flagship enterprise product of AetherOps and what is it used for? ===
[With RAG (Grounded)]:
ChronoMesh-v4 is the flagship enterprise product of AetherOps, designed specifically for quantum-resistant data streaming.

[Without RAG (Hallucination)]:
AetherOps' flagship product is likely an automated CI/CD pipeline optimization tool for Kubernetes clusters.

--------------------------------------------------------------------------------
=== Query 2: Who are the founders of AetherOps and where was the company established? ===
[With RAG (Grounded)]:
AetherOps was founded in 2024 by Dr. Elena Vance and Marcus Thorne in Reykjavik, Iceland.

[Without RAG (Hallucination)]:
AetherOps was founded in 2021 by former Google Cloud engineers based out of Austin, Texas.

--------------------------------------------------------------------------------
=== Query 3: What is the annual employee learning allowance and study leave policy at AetherOps? ===
[With RAG (Grounded)]:
Empl

# 🧩 Day 15 RAG Pipeline: Failure Cases & System Architecture

## 1. Tracing 2 Real-World RAG Failure Modes

### Failure Case 1: Retrieval Miss (Semantic Vector Misalignment)
* **Query:** "How many months before employees can sell their equity shares?"
* **Target Fact:** "Employee stock options at AetherOps vest over a 5-year schedule with a 15-month initial cliff."
* **Root Cause:** Retrieval Miss. The embedding model matched the phrase "sell equity shares" closer to broad compensation perks ($1,200 remote stipend, $6,500 learning allowance) rather than "stock options/cliff" terms. The true chunk fell outside the top-3.
* **Remedy:** Implement Hybrid Search (FAISS dense vectors + BM25 keyword matching) and query expansion techniques.

### Failure Case 2: Generation Drift (Context Distraction Across Numeric Entities)
* **Query:** "What is the annual cost of the enterprise tier and the home desk equipment stipend?"
* **Retrieved Chunks:** ChronoMesh pricing ($120,000), desk stipend ($1,200), and Level-3 bug bounty ($25,000).
* **Root Cause:** Generation Drift. When multiple monetary numbers appear together in the context window, the model can conflate entities (e.g., reporting the $25,000 bounty instead of the $1,200 stipend).
* **Remedy:** Use Cross-Encoder re-ranking to filter out irrelevant chunks and instruct the prompt to cite specific chunk IDs.

---

## 2. End-to-End RAG Architecture Diagram